# Preprocessing

In [ ]:
import warnings
import pandas as pd
warnings.filterwarnings("ignore")

In [ ]:
%run ./agg_spark

In [ ]:
# meter data
meter_data = spark.read.parquet("abfss://902275ba-e699-4505-9532-1f979686157f@onelake.dfs.fabric.microsoft.com/7e0f80c1-43ae-46aa-a652-f7b65014d652/Tables/GNM63V")
display(meter_data)

In [ ]:
# teriff data
tariff_df = (
    spark.read.format("csv")
    .option("header", "true")
    .option("sep", ";")
    .load("abfss://902275ba-e699-4505-9532-1f979686157f@onelake.dfs.fabric.microsoft.com/cd3a1398-1ea6-4b6b-a5c7-68d125bf9ac0/Files/Anläggningar med tidsindelad from 1 dec 2025.csv")
)
tariff_df.createOrReplaceTempView("tariff_df")
display(tariff_df)

In [ ]:
# Electricity meter data
# print("Unique households:", len(meter_data["aID"].unique())) # unique households
# print("Unique days:", len(meter_data["TIDPUNKT_DAG"].unique())) # unique days
# display(meter_data.head(3))


# Tariff data
# print(f"tariff rows: {len(tariff_df):,}") # Tariff number 
# display(tariff_df.head(3))



# Electricity meter data
print("Unique households:", meter_data.select("aID").distinct().count()) # unique households
print("Unique days:", meter_data.select("TIDPUNKT_DAG").distinct().count()) # unique days


# Tariff data
print(f"tariff rows: {tariff_df.count()}") # Tariff number 

## Aggregate electricity meter data

In [ ]:
agg = ElectricityAggregator(meter_data, tariff_df)

In [ ]:
# Monthly
month_result = agg.run(
    freq="month",
    agg_method=["top3_mean", "variance", "mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    table_name="meter_monthly"
    )
    
# month_result.head()

In [ ]:
# try if successfully convert to pandas, use the original code
month_data = month_result.toPandas()
tariff_data = tariff_df.toPandas()

In [ ]:
month_data.head()

In [ ]:
# save monthly data
month_data.to_parquet("/lakehouse/default/Files/month_data", index=False)

In [ ]:
pd.set_option('display.max_rows', None)
chunk_size = 5000

In [ ]:
month_result_high = month_result[month_result["price"] == "high"]

for i in range(0, len(month_result_high), chunk_size):
    print(f"Rows {i} to {i+chunk_size}")
    display(month_result_high.iloc[i:i+chunk_size])

In [ ]:
month_result_all = month_result[month_result["price"] == "all"]

for i in range(0, len(month_result_all), chunk_size):
    print(f"Rows {i} to {i+chunk_size}")
    display(month_result_all.iloc[i:i+chunk_size])

In [ ]:
month_result_low = month_result[month_result["price"] == "low"]

for i in range(0, len(month_result_low), chunk_size):
    print(f"Rows {i} to {i+chunk_size}")
    display(month_result_low.iloc[i:i+chunk_size])

In [ ]:
# Hourly
hour_result = agg.run(
    freq="hour",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    )
# hour_result.head(3)

hour_data = hour_result.toPandas()
hour_data.to_parquet("/lakehouse/default/Files/hour_data", index=False)

In [ ]:
# Week & Weekend
weekday_result = agg.run(
    freq="weekday",
    agg_method=["mean"],
    use_price=True,
    add_user_group_col=[0.25, 0.75],
    )
weekday_data = weekday_result.toPandas()
weekday_data.to_parquet("/lakehouse/default/Files/weekday_data", index=False)